<a href="https://colab.research.google.com/github/safiamussaratt/landsat-uhi-predictor/blob/main/ds_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Colour palette ───────────────────────────────────────────────────────────
P = {
    "bg":     "#0F1117",
    "panel":  "#1A1D2E",
    "accent": "#6C63FF",
    "warm":   "#FF6B6B",
    "cool":   "#4ECDC4",
    "gold":   "#FFD166",
    "green":  "#06D6A0",
    "text":   "#E8E8F0",
    "subtle": "#888899",
}
MC = ["#6C63FF", "#FF6B6B", "#FFD166", "#4ECDC4", "#06D6A0"]

def sax(ax, title="", xl="", yl="")
    ax.set_facecolor(P["panel"])
    ax.set_title(title, color=P["text"], fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel(xl, color=P["subtle"], fontsize=8)
    ax.set_ylabel(yl, color=P["subtle"], fontsize=8)
    ax.tick_params(colors=P["subtle"], labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor(P["panel"])

In [6]:
# 1. DATA LOADING & PREPROCESSING
print("====== DATA LOADING & PREPROCESSING ======\n")

CSV_PATH = "UHI_Islamabad_Landsat9.csv"
df_raw = pd.read_csv(CSV_PATH)
print(f"Raw shape: {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")

# ── Extract coordinates from GeoJSON .geo column ──────────────────────────────
def parse_geo(geo_str):
    try:
        coords = json.loads(geo_str)["coordinates"]
        return pd.Series({"lon": coords[0], "lat": coords[1]})
    except Exception:
        return pd.Series({"lon": np.nan, "lat": np.nan})

geo = df_raw[".geo"].apply(parse_geo)
df  = pd.concat([df_raw.drop(columns=["system:index", ".geo"]), geo], axis=1)

# ── Missing value handling ────────────────────────────────────────────────────
print(f"\nMissing values:\n{df.isnull().sum().to_string()}")
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\nClean shape: {df.shape}")

# ── Feature Engineering ───────────────────────────────────────────────────────
df["NDBI_NDVI_ratio"] = df["NDBI"] / (df["NDVI"] + 1e-6)   # thermal-veg contrast
df["VegWater"]        = df["NDVI"] * df["NDWI"]             # vegetation × moisture
df["BuiltStress"]     = df["NDBI"] - df["NDVI"]             # urbanisation stress index
df["Aridity"]         = ((df["NDBI"] - df["NDWI"])          # aridity / dryness proxy
                         / (df["NDBI"] + df["NDWI"] + 1e-6))
df["lon_lat"]         = df["lon"] * df["lat"]               # spatial interaction term
df["NDBI_sq"]         = df["NDBI"] ** 2                     # non-linear NDBI effect
df["NDVI_sq"]         = df["NDVI"] ** 2                     # non-linear NDVI effect

FEATURES = [
    "NDBI", "NDVI", "NDWI",
    "lon", "lat",
    "NDBI_NDVI_ratio", "VegWater", "BuiltStress",
    "Aridity", "lon_lat", "NDBI_sq", "NDVI_sq",
]
TARGET = "LST"

print(f"\nFeature set ({len(FEATURES)}): {FEATURES}")
print(f"\nDescriptive statistics:")
print(df[["LST", "NDBI", "NDVI", "NDWI"]].describe().round(4).to_string())

# ── UHI binary label (top 30 % LST = UHI hotspot) ────────────────────────────
UHI_THRESH      = np.percentile(df[TARGET], 70)
df["UHI_zone"]  = (df[TARGET] >= UHI_THRESH).astype(int)
print(f"\nUHI threshold (70th pct): {UHI_THRESH:.2f} °C")
print(f"UHI pixels: {df['UHI_zone'].sum()} / {len(df)} ({df['UHI_zone'].mean()*100:.1f}%)")

# ── Normalise & split ─────────────────────────────────────────────────────────
X_raw = df[FEATURES].values
y     = df[TARGET].values

scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
df_te_raw = df.iloc[
    train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)[1]
].reset_index(drop=True)

print(f"\nTrain: {X_tr.shape[0]} samples | Test: {X_te.shape[0]} samples")

====== DATA LOADING & PREPROCESSING ======

Raw shape: (5000, 6)
Columns  : ['system:index', 'LST', 'NDBI', 'NDVI', 'NDWI', '.geo']

Missing values:
LST     0
NDBI    0
NDVI    0
NDWI    0
lon     0
lat     0

Clean shape: (5000, 6)

Feature set (12): ['NDBI', 'NDVI', 'NDWI', 'lon', 'lat', 'NDBI_NDVI_ratio', 'VegWater', 'BuiltStress', 'Aridity', 'lon_lat', 'NDBI_sq', 'NDVI_sq']

Descriptive statistics:
             LST       NDBI       NDVI       NDWI
count  5000.0000  5000.0000  5000.0000  5000.0000
mean     31.3062    -0.0439     0.1885    -0.2021
std       3.1323     0.0474     0.0664     0.0539
min      16.3511    -0.2016    -0.0439    -0.3804
25%      29.6353    -0.0737     0.1432    -0.2359
50%      31.8433    -0.0394     0.1895    -0.2071
75%      33.5224    -0.0095     0.2316    -0.1719
max      39.5697     0.0820     0.4029     0.0839

UHI threshold (70th pct): 33.15 °C
UHI pixels: 1500 / 5000 (30.0%)

Train: 4000 samples | Test: 1000 samples


In [7]:
# 2. EXPLORATORY DATA ANALYSIS

print("===== EXPLORATORY DATA ANALYSIS ======\n")

fig, axes = plt.subplots(2, 4, figsize=(20, 10), facecolor=P["bg"])
axes = axes.flatten()

# (a) LST distribution
axes[0].hist(df["LST"], bins=55, color=P["warm"], edgecolor="none", alpha=0.85)
axes[0].axvline(UHI_THRESH, color=P["gold"], lw=2, ls="--",
                label=f"UHI ≥ {UHI_THRESH:.1f}°C")
axes[0].legend(fontsize=7, facecolor=P["panel"], labelcolor=P["text"])
sax(axes[0], "LST Distribution", "Temperature (°C)", "Count")

# (b) NDVI vs LST — CORE RESEARCH QUESTION PLOT 1
sc = axes[1].scatter(df["NDVI"], df["LST"], c=df["LST"], cmap="RdYlGn_r",
                     s=5, alpha=0.5)
plt.colorbar(sc, ax=axes[1]).ax.tick_params(colors=P["subtle"])
# Add regression line
m, b = np.polyfit(df["NDVI"], df["LST"], 1)
xline = np.linspace(df["NDVI"].min(), df["NDVI"].max(), 200)
axes[1].plot(xline, m * xline + b, color=P["gold"], lw=2, ls="--",
             label=f"y={m:.2f}x+{b:.1f}")
r_ndvi = df[["NDVI","LST"]].corr().iloc[0,1]
axes[1].text(0.05, 0.93, f"r = {r_ndvi:.3f}", transform=axes[1].transAxes,
             fontsize=9, color=P["gold"], fontweight="bold")
axes[1].legend(fontsize=7, facecolor=P["panel"], labelcolor=P["text"])
sax(axes[1], "NDVI vs LST  [Core RQ]", "NDVI", "LST (°C)")

# (c) NDBI vs LST — CORE RESEARCH QUESTION PLOT 2
sc = axes[2].scatter(df["NDBI"], df["LST"], c=df["LST"], cmap="plasma",
                     s=5, alpha=0.5)
plt.colorbar(sc, ax=axes[2]).ax.tick_params(colors=P["subtle"])
m2, b2 = np.polyfit(df["NDBI"], df["LST"], 1)
xline2 = np.linspace(df["NDBI"].min(), df["NDBI"].max(), 200)
axes[2].plot(xline2, m2 * xline2 + b2, color=P["gold"], lw=2, ls="--",
             label=f"y={m2:.2f}x+{b2:.1f}")
r_ndbi = df[["NDBI","LST"]].corr().iloc[0,1]
axes[2].text(0.05, 0.93, f"r = {r_ndbi:.3f}", transform=axes[2].transAxes,
             fontsize=9, color=P["gold"], fontweight="bold")
axes[2].legend(fontsize=7, facecolor=P["panel"], labelcolor=P["text"])
sax(axes[2], "NDBI vs LST  [Core RQ]", "NDBI", "LST (°C)")

# (d) NDWI vs LST
sc = axes[3].scatter(df["NDWI"], df["LST"], c=df["LST"], cmap="cool",
                     s=5, alpha=0.5)
plt.colorbar(sc, ax=axes[3]).ax.tick_params(colors=P["subtle"])
r_ndwi = df[["NDWI","LST"]].corr().iloc[0,1]
axes[3].text(0.05, 0.93, f"r = {r_ndwi:.3f}", transform=axes[3].transAxes,
             fontsize=9, color=P["gold"], fontweight="bold")
sax(axes[3], "NDWI vs LST", "NDWI", "LST (°C)")

# (e) Spatial LST map
sc = axes[4].scatter(df["lon"], df["lat"], c=df["LST"], cmap="plasma",
                     s=5, alpha=0.75)
cb = plt.colorbar(sc, ax=axes[4])
cb.ax.tick_params(colors=P["subtle"])
cb.set_label("LST (°C)", color=P["subtle"], fontsize=8)
sax(axes[4], "Spatial LST Distribution", "Longitude", "Latitude")

# (f) Box: LST by UHI zone
bp = axes[5].boxplot(
    [df.loc[df["UHI_zone"]==0, "LST"], df.loc[df["UHI_zone"]==1, "LST"]],
    patch_artist=True,
    medianprops=dict(color=P["gold"], lw=2),
)
bp["boxes"][0].set_facecolor(P["cool"])
bp["boxes"][1].set_facecolor(P["warm"])
for w in bp["whiskers"] + bp["caps"] + bp["fliers"]:
    w.set(color=P["subtle"], markersize=2)
axes[5].set_xticklabels(["Non-UHI", "UHI"], color=P["text"])
sax(axes[5], "LST by UHI Zone", "", "LST (°C)")

# (g) Correlation heatmap
corr_cols = ["LST","NDBI","NDVI","NDWI","BuiltStress","Aridity","VegWater"]
corr = df[corr_cols].corr()
sns.heatmap(corr, ax=axes[6], cmap="RdBu_r", center=0, annot=True,
            fmt=".2f", annot_kws={"size": 7},
            linewidths=0.3, linecolor=P["bg"])
axes[6].set_facecolor(P["panel"])
axes[6].set_title("Correlation Matrix", color=P["text"], fontsize=10,
                  fontweight="bold", pad=8)
axes[6].tick_params(colors=P["subtle"], labelsize=7)

# (h) Feature |r| with LST
feat_corr = df[FEATURES+[TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values()
axes[7].barh(feat_corr.index, feat_corr.values,
             color=[P["accent"] if v > 0.3 else P["subtle"]
                    for v in feat_corr.values])
axes[7].axvline(0.3, color=P["gold"], ls="--", lw=1, label="|r|=0.3")
axes[7].legend(fontsize=7, facecolor=P["panel"], labelcolor=P["text"])
sax(axes[7], "Feature Correlation with LST", "|Pearson r|", "")

fig.suptitle("UHI Islamabad — Exploratory Data Analysis (Landsat-9)",
             color=P["text"], fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_eda.png", dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print("EDA figure saved → fig_eda.png")

===== EXPLORATORY DATA ANALYSIS ======

EDA figure saved → fig_eda.png


In [9]:
# 3. SKLEARN MODELS (LR, RF, GBR, MLP)

print("====== SKLEARN MODELS ======")

def evaluate_sklearn(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    yp   = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, yp))
    mae  = mean_absolute_error(yte, yp)
    r2   = r2_score(yte, yp)
    mape = mean_absolute_percentage_error(yte, yp) * 100
    cv   = -cross_val_score(model, Xtr, ytr, cv=5,
                             scoring="neg_root_mean_squared_error").mean()
    print(f"  {name:<28}  RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"R²={r2:.4f}  MAPE={mape:.3f}%  CV-RMSE={cv:.4f}")
    return {"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2,
            "MAPE": mape, "CV_RMSE": cv, "yp": yp}

sklearn_results = []

# 1. Linear Regression
print("\n[1] Linear Regression")
sklearn_results.append(
    evaluate_sklearn("Linear Regression", LinearRegression(),
                     X_tr, y_tr, X_te, y_te)
)

# 2. Random Forest — hyperparameter tuning
print("\n[2] Random Forest")
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [10, 20, None],
     "min_samples_split": [2, 5]},
    cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
rf_gs.fit(X_tr, y_tr)
print(f"  Best params: {rf_gs.best_params_}")
sklearn_results.append(
    evaluate_sklearn("Random Forest", rf_gs.best_estimator_,
                     X_tr, y_tr, X_te, y_te)
)
rf_model = rf_gs.best_estimator_

# 3. Gradient Boosting — hyperparameter tuning
print("\n[3] Gradient Boosting (XGBoost-equivalent)")
gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=SEED),
    {"n_estimators": [200, 400], "learning_rate": [0.05, 0.1],
     "max_depth": [3, 5], "subsample": [0.8, 1.0]},
    cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
gb_gs.fit(X_tr, y_tr)
print(f"  Best params: {gb_gs.best_params_}")
sklearn_results.append(
    evaluate_sklearn("Gradient Boosting", gb_gs.best_estimator_,
                     X_tr, y_tr, X_te, y_te)
)

# 4. MLP — hyperparameter tuning
print("\n[4] MLP Neural Network")
mlp_gs = GridSearchCV(
    MLPRegressor(max_iter=500, early_stopping=True,
                 validation_fraction=0.1, random_state=SEED),
    {"hidden_layer_sizes": [(128, 64, 32), (256, 128, 64)],
     "activation": ["relu", "tanh"],
     "alpha": [1e-4, 1e-3],
     "learning_rate_init": [1e-3, 5e-4]},
    cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
mlp_gs.fit(X_tr, y_tr)
print(f"  Best params: {mlp_gs.best_params_}")
sklearn_results.append(
    evaluate_sklearn("MLP", mlp_gs.best_estimator_,
                     X_tr, y_tr, X_te, y_te)
)

====== SKLEARN MODELS ======

[1] Linear Regression
  Linear Regression             RMSE=1.7214  MAE=1.3381  R²=0.7032  MAPE=4.416%  CV-RMSE=1.7145

[2] Random Forest
  Best params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}
  Random Forest                 RMSE=1.2973  MAE=1.0114  R²=0.8315  MAPE=3.280%  CV-RMSE=1.3118

[3] Gradient Boosting (XGBoost-equivalent)
  Best params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 400, 'subsample': 0.8}
  Gradient Boosting             RMSE=1.2576  MAE=0.9738  R²=0.8416  MAPE=3.152%  CV-RMSE=1.2892

[4] MLP Neural Network
  Best params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate_init': 0.0005}
  MLP                           RMSE=1.4585  MAE=1.1400  R²=0.7870  MAPE=3.710%  CV-RMSE=1.5113


In [10]:
# 4. TRUE 1D-CNN (PyTorch)
print("====== 1D-CNN (PyTorch) ======\n")

class UHI_CNN1D(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.conv_block = nn.Sequential(
            # Conv layer 1
            nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            # Conv layer 2
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            # Conv layer 3
            nn.Conv1d(in_channels=128, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)   # → (batch, 64, 1)
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # x: (batch, n_features) → (batch, 1, n_features)
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        x = self.global_avg_pool(x).squeeze(-1)   # (batch, 64)
        return self.fc(x).squeeze(-1)             # (batch,)


def train_cnn(X_train, y_train, X_val, y_val,
              n_epochs=150, batch_size=256, lr=1e-3, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}")

    # Convert to tensors
    Xtr_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    Xvl_t = torch.tensor(X_val,   dtype=torch.float32).to(device)
    yvl_t = torch.tensor(y_val,   dtype=torch.float32).to(device)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                        batch_size=batch_size, shuffle=True)

    model     = UHI_CNN1D(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5,
                                                     factor=0.5)

    best_val_loss = float("inf")
    best_weights  = None
    no_improve    = 0
    train_losses, val_losses = [], []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader:
            optimizer.zero_grad()
            pred = model(Xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(yb)
        epoch_loss /= len(y_train)

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xvl_t), yvl_t).item()
        scheduler.step(val_loss)
        train_losses.append(epoch_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:>3}/{n_epochs}  "
                  f"train_loss={epoch_loss:.4f}  val_loss={val_loss:.4f}")

    model.load_state_dict(best_weights)
    return model, device, train_losses, val_losses


def predict_cnn(model, device, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(X, dtype=torch.float32).to(device)
        return model(Xt).cpu().numpy()


# Hyperparameter search for CNN (learning rate & batch size)
print("\n  Hyperparameter search: lr × batch_size")
# Use 80/20 of training data for CNN validation
X_cnn_tr, X_cnn_val, y_cnn_tr, y_cnn_val = train_test_split(
    X_tr, y_tr, test_size=0.15, random_state=SEED
)

best_cnn_rmse = float("inf")
best_cnn_cfg  = {}
best_cnn_model, best_cnn_device = None, None
cnn_train_losses, cnn_val_losses = [], []

for lr in [1e-3, 5e-4]:
    for bs in [256, 512]:
        print(f"\n  Trying lr={lr}, batch_size={bs}")
        m, dev, tl, vl = train_cnn(
            X_cnn_tr, y_cnn_tr, X_cnn_val, y_cnn_val,
            n_epochs=150, batch_size=bs, lr=lr, patience=15,
        )
        yp_val = predict_cnn(m, dev, X_cnn_val)
        rmse   = np.sqrt(mean_squared_error(y_cnn_val, yp_val))
        print(f"  Val RMSE = {rmse:.4f}")
        if rmse < best_cnn_rmse:
            best_cnn_rmse  = rmse
            best_cnn_cfg   = {"lr": lr, "batch_size": bs}
            best_cnn_model = m
            best_cnn_device = dev
            cnn_train_losses, cnn_val_losses = tl, vl

print(f"\n  Best CNN config: {best_cnn_cfg}  |  Val RMSE: {best_cnn_rmse:.4f}")

# Final evaluation on held-out test set
cnn_yp   = predict_cnn(best_cnn_model, best_cnn_device, X_te)
cnn_rmse = np.sqrt(mean_squared_error(y_te, cnn_yp))
cnn_mae  = mean_absolute_error(y_te, cnn_yp)
cnn_r2   = r2_score(y_te, cnn_yp)
cnn_mape = mean_absolute_percentage_error(y_te, cnn_yp) * 100
print(f"\n  1D-CNN (PyTorch)             "
      f"RMSE={cnn_rmse:.4f}  MAE={cnn_mae:.4f}  "
      f"R²={cnn_r2:.4f}  MAPE={cnn_mape:.3f}%")

cnn_result = {
    "Model": "1D-CNN (PyTorch)",
    "RMSE": cnn_rmse, "MAE": cnn_mae,
    "R2": cnn_r2, "MAPE": cnn_mape,
    "CV_RMSE": None, "yp": cnn_yp,
}

====== 1D-CNN (PyTorch) ======


  Hyperparameter search: lr × batch_size

  Trying lr=0.001, batch_size=256
  Device: cpu
  Epoch  20/150  train_loss=536.5052  val_loss=472.9157
  Epoch  40/150  train_loss=97.7276  val_loss=67.5945
  Epoch  60/150  train_loss=23.5298  val_loss=9.6022
  Epoch  80/150  train_loss=19.7835  val_loss=5.4869
  Epoch 100/150  train_loss=20.3796  val_loss=5.4272
  Early stopping at epoch 108
  Val RMSE = 1.8526

  Trying lr=0.001, batch_size=512
  Device: cpu
  Epoch  20/150  train_loss=798.9448  val_loss=786.7413
  Epoch  40/150  train_loss=545.2087  val_loss=563.3688
  Epoch  60/150  train_loss=274.4959  val_loss=276.0772
  Epoch  80/150  train_loss=99.2557  val_loss=108.4790
  Epoch 100/150  train_loss=37.7931  val_loss=22.6028
  Epoch 120/150  train_loss=25.0279  val_loss=10.6570
  Epoch 140/150  train_loss=23.0543  val_loss=6.5806
  Early stopping at epoch 141
  Val RMSE = 2.2572

  Trying lr=0.0005, batch_size=256
  Device: cpu
  Epoch  20/150  train_lo

In [11]:
# 5. EVALUATION & COMPARATIVE ANALYSIS
print("====== COMPARATIVE EVALUATION ======\n")

all_results = sklearn_results + [cnn_result]
res_df      = pd.DataFrame([{k: v for k, v in r.items() if k != "yp"}
                             for r in all_results])
res_df_sorted = res_df.sort_values("RMSE")
print("\nRanked by RMSE (ascending = better):")
print(res_df_sorted.to_string(index=False))

# ── Figure 2: Evaluation dashboard ───────────────────────────────────────────
fig = plt.figure(figsize=(22, 16), facecolor=P["bg"])
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.40)

model_order = [r["Model"] for r in all_results]

# Scatter plots (actual vs predicted) — one per model
positions = [(0,0),(0,1),(0,2),(0,3),(1,0)]
for idx, r in enumerate(all_results):
    row, col = positions[idx]
    ax = fig.add_subplot(gs[row, col])
    mn, mx = y_te.min(), y_te.max()
    ax.scatter(y_te, r["yp"], s=5, alpha=0.35, color=MC[idx])
    ax.plot([mn,mx],[mn,mx], color=P["gold"], lw=1.5, ls="--")
    ax.text(0.05, 0.92,
            f"R²  = {r['R2']:.4f}\nRMSE = {r['RMSE']:.3f} °C",
            transform=ax.transAxes, fontsize=8, color=P["text"], va="top",
            bbox=dict(facecolor=P["bg"], alpha=0.65, pad=3))
    sax(ax, r["Model"], "Actual LST (°C)", "Predicted (°C)")

# RMSE bar
ax6 = fig.add_subplot(gs[1, 1])
rmses = [r["RMSE"] for r in all_results]
bars  = ax6.bar(range(len(model_order)), rmses, color=MC, width=0.65)
best  = np.argmin(rmses)
bars[best].set_edgecolor(P["gold"]); bars[best].set_linewidth(2.5)
for b, v in zip(bars, rmses):
    ax6.text(b.get_x()+b.get_width()/2, v+0.01,
             f"{v:.3f}", ha="center", va="bottom", color=P["text"], fontsize=8)
ax6.set_xticks(range(len(model_order)))
ax6.set_xticklabels(["LR","RF","GBR","MLP","CNN"], color=P["text"])
sax(ax6, "RMSE Comparison ↓", "Model", "RMSE (°C)")

# R² bar
ax7 = fig.add_subplot(gs[1, 2])
r2s  = [r["R2"] for r in all_results]
bars7 = ax7.bar(range(len(model_order)), r2s, color=MC, width=0.65)
best7 = np.argmax(r2s)
bars7[best7].set_edgecolor(P["gold"]); bars7[best7].set_linewidth(2.5)
for b, v in zip(bars7, r2s):
    ax7.text(b.get_x()+b.get_width()/2, v+0.002,
             f"{v:.4f}", ha="center", va="bottom", color=P["text"], fontsize=7)
ax7.set_xticks(range(len(model_order)))
ax7.set_xticklabels(["LR","RF","GBR","MLP","CNN"], color=P["text"])
sax(ax7, "R² Score ↑", "Model", "R²")

# Residual distribution
ax8 = fig.add_subplot(gs[1, 3])
for idx, r in enumerate(all_results):
    ax8.hist(y_te - r["yp"], bins=45, alpha=0.5,
             color=MC[idx], label=r["Model"].split()[0], edgecolor="none")
ax8.axvline(0, color=P["gold"], lw=1.5, ls="--")
ax8.legend(fontsize=6.5, facecolor=P["panel"], labelcolor=P["text"])
sax(ax8, "Residual Distribution", "Residual (°C)", "Count")

# CNN training curve
ax9 = fig.add_subplot(gs[2, 0:2])
ax9.plot(cnn_train_losses, color=P["accent"], lw=1.8, label="Train loss")
ax9.plot(cnn_val_losses,   color=P["warm"],   lw=1.8, label="Val loss")
ax9.legend(fontsize=8, facecolor=P["panel"], labelcolor=P["text"])
sax(ax9, "1D-CNN Training Curve (MSE loss)", "Epoch", "MSE Loss")

# Feature importance (RF)
ax10 = fig.add_subplot(gs[2, 2])
fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
ax10.barh(fi.index, fi.values,
          color=[P["accent"] if v > 0.05 else P["subtle"] for v in fi.values])
ax10.axvline(0.05, color=P["gold"], ls="--", lw=1)
sax(ax10, "RF Feature Importance", "Importance", "")

# Summary table
ax11 = fig.add_subplot(gs[2, 3])
ax11.axis("off")
table_data = [[r["Model"],
               f"{r['RMSE']:.4f}",
               f"{r['MAE']:.4f}",
               f"{r['R2']:.4f}",
               f"{r['MAPE']:.3f}%"]
              for r in sorted(all_results, key=lambda x: x["RMSE"])]
tbl = ax11.table(
    cellText=table_data,
    colLabels=["Model", "RMSE", "MAE", "R²", "MAPE"],
    loc="center", cellLoc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(7.5)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor(P["accent"] if row == 0 else P["panel"])
    cell.set_text_props(color=P["text"])
    cell.set_edgecolor(P["bg"])
ax11.set_title("Ranked Model Summary", color=P["text"], fontsize=10,
               fontweight="bold", pad=8)

fig.suptitle("UHI Islamabad — ML Evaluation & Comparative Analysis (Landsat-9)",
             color=P["text"], fontsize=14, fontweight="bold")
plt.savefig("fig_eval.png", dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print("\nEvaluation figure saved → fig_eval.png")

====== COMPARATIVE EVALUATION ======


Ranked by RMSE (ascending = better):
            Model     RMSE      MAE       R2     MAPE  CV_RMSE
Gradient Boosting 1.257618 0.973766 0.841612 3.152390 1.289162
    Random Forest 1.297283 1.011393 0.831464 3.280449 1.311754
              MLP 1.458494 1.139987 0.786974 3.709696 1.511292
 1D-CNN (PyTorch) 1.688865 1.314897 0.714363 4.171184      NaN
Linear Regression 1.721434 1.338111 0.703240 4.415952 1.714524

Evaluation figure saved → fig_eval.png


In [12]:
# 6. UHI HOTSPOT MAP
print("====== UHI HOTSPOT MAP ======\n")

# Use CNN predictions on the full dataset for the hotspot map
all_X = scaler.transform(df[FEATURES].values)
cnn_all_yp = predict_cnn(best_cnn_model, best_cnn_device, all_X)

fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=P["bg"])

# (a) Actual LST spatial map
sc1 = axes[0].scatter(df["lon"], df["lat"], c=df["LST"],
                      cmap="plasma", s=8, alpha=0.8)
cb1 = plt.colorbar(sc1, ax=axes[0])
cb1.ax.tick_params(colors=P["subtle"])
cb1.set_label("LST (°C)", color=P["subtle"])
sax(axes[0], "Actual LST — Spatial Map", "Longitude", "Latitude")

# (b) CNN-predicted LST spatial map
sc2 = axes[1].scatter(df["lon"], df["lat"], c=cnn_all_yp,
                      cmap="plasma", s=8, alpha=0.8,
                      vmin=df["LST"].min(), vmax=df["LST"].max())
cb2 = plt.colorbar(sc2, ax=axes[1])
cb2.ax.tick_params(colors=P["subtle"])
cb2.set_label("LST (°C)", color=P["subtle"])
sax(axes[1], "CNN-Predicted LST — Spatial Map", "Longitude", "Latitude")

# (c) UHI Hotspot classification
# Non-UHI = cool blue, UHI = hot red
cmap_uhi   = mcolors.ListedColormap([P["cool"], P["warm"]])
pred_label = (cnn_all_yp >= UHI_THRESH).astype(int)
sc3 = axes[2].scatter(df["lon"], df["lat"], c=pred_label,
                      cmap=cmap_uhi, s=8, alpha=0.85, vmin=0, vmax=1)
cb3 = plt.colorbar(sc3, ax=axes[2], ticks=[0.25, 0.75])
cb3.ax.set_yticklabels(["Non-UHI", "UHI"], color=P["text"], fontsize=8)
cb3.ax.tick_params(colors=P["subtle"])
# Overlay actual hotspot boundary
uhi_pts = df[df["UHI_zone"] == 1]
axes[2].scatter(uhi_pts["lon"], uhi_pts["lat"], facecolors="none",
                edgecolors=P["gold"], s=12, linewidths=0.4,
                alpha=0.4, label="Actual UHI boundary")
axes[2].legend(fontsize=7, facecolor=P["panel"], labelcolor=P["text"])
sax(axes[2], f"UHI Hotspot Map (threshold ≥ {UHI_THRESH:.1f}°C)",
    "Longitude", "Latitude")

fig.suptitle("UHI Islamabad — Spatial Hotspot Analysis (CNN Predictions vs Actual)",
             color=P["text"], fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_hotspot.png", dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print("Hotspot map saved → fig_hotspot.png")

====== UHI HOTSPOT MAP ======

Hotspot map saved → fig_hotspot.png


In [13]:
# 7. CORE RESEARCH QUESTION — ISOLATION ANALYSIS
# =============================================================================
print("\n" + "=" * 65)
print("=== NDVI & NDBI ISOLATION ANALYSIS ===")
print("=" * 65)

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=P["bg"])

# (a) NDVI vs LST — binned mean
ax = axes[0, 0]
bins = pd.cut(df["NDVI"], bins=20)
mean_lst = df.groupby(bins, observed=True)["LST"].mean()
bin_centres = [iv.mid for iv in mean_lst.index]
ax.bar(range(len(bin_centres)), mean_lst.values, color=P["green"], alpha=0.8, width=0.8)
ax.set_xticks(range(0, len(bin_centres), 4))
ax.set_xticklabels([f"{bin_centres[i]:.2f}" for i in range(0,len(bin_centres),4)], rotation=30)
sax(ax, "Mean LST per NDVI Bin", "NDVI", "Mean LST (°C)")

# (b) NDBI vs LST — binned mean
ax = axes[0, 1]
bins2 = pd.cut(df["NDBI"], bins=20)
mean_lst2 = df.groupby(bins2, observed=True)["LST"].mean()
bin_centres2 = [iv.mid for iv in mean_lst2.index]
ax.bar(range(len(bin_centres2)), mean_lst2.values, color=P["warm"], alpha=0.8, width=0.8)
ax.set_xticks(range(0, len(bin_centres2), 4))
ax.set_xticklabels([f"{bin_centres2[i]:.3f}" for i in range(0,len(bin_centres2),4)], rotation=30)
sax(ax, "Mean LST per NDBI Bin", "NDBI", "Mean LST (°C)")

# (c) NDVI quantile boxes vs LST
ax = axes[0, 2]
df["NDVI_q"] = pd.qcut(df["NDVI"], q=5, labels=["Q1\n(Low)", "Q2", "Q3", "Q4", "Q5\n(High)"])
groups = [df.loc[df["NDVI_q"]==q, "LST"].values for q in df["NDVI_q"].cat.categories]
bp = ax.boxplot(groups, patch_artist=True,
                medianprops=dict(color=P["gold"], lw=2))
colors_box = [P["warm"], "#FF9A6B", P["subtle"], "#6BE0D4", P["cool"]]
for box, col in zip(bp["boxes"], colors_box):
    box.set_facecolor(col)
for w in bp["whiskers"]+bp["caps"]+bp["fliers"]:
    w.set(color=P["subtle"], markersize=2)
ax.set_xticklabels(["Q1\n(Low)","Q2","Q3","Q4","Q5\n(High)"], color=P["text"])
sax(ax, "LST by NDVI Quantile\n(Q1=sparse vegetation, Q5=dense)", "NDVI Quintile", "LST (°C)")

# (d) NDBI quantile boxes vs LST
ax = axes[1, 0]
df["NDBI_q"] = pd.qcut(df["NDBI"], q=5, labels=["Q1\n(Low)", "Q2", "Q3", "Q4", "Q5\n(High)"])
groups2 = [df.loc[df["NDBI_q"]==q, "LST"].values for q in df["NDBI_q"].cat.categories]
bp2 = ax.boxplot(groups2, patch_artist=True,
                 medianprops=dict(color=P["gold"], lw=2))
colors_box2 = [P["cool"], "#6BE0D4", P["subtle"], "#FF9A6B", P["warm"]]
for box, col in zip(bp2["boxes"], colors_box2):
    box.set_facecolor(col)
for w in bp2["whiskers"]+bp2["caps"]+bp2["fliers"]:
    w.set(color=P["subtle"], markersize=2)
ax.set_xticklabels(["Q1\n(Low)","Q2","Q3","Q4","Q5\n(High)"], color=P["text"])
sax(ax, "LST by NDBI Quantile\n(Q1=low built-up, Q5=high built-up)", "NDBI Quintile", "LST (°C)")

# (e) 2D density: NDVI × NDBI coloured by LST
ax = axes[1, 1]
sc = ax.scatter(df["NDVI"], df["NDBI"], c=df["LST"], cmap="plasma",
                s=5, alpha=0.6)
cb = plt.colorbar(sc, ax=ax)
cb.set_label("LST (°C)", color=P["subtle"])
cb.ax.tick_params(colors=P["subtle"])
# Mark the four quadrants
ax.axvline(df["NDVI"].median(), color=P["gold"], lw=1, ls="--", alpha=0.6)
ax.axhline(df["NDBI"].median(), color=P["gold"], lw=1, ls="--", alpha=0.6)
ax.text(0.72, 0.92, "Urban\nheat", transform=ax.transAxes,
        fontsize=7, color=P["warm"], fontweight="bold")
ax.text(0.05, 0.08, "Vegetation\ncool", transform=ax.transAxes,
        fontsize=7, color=P["cool"], fontweight="bold")
sax(ax, "NDVI × NDBI Index Space (coloured by LST)",
    "NDVI", "NDBI")

# (f) Partial dependence proxy: CNN predicted LST vs NDVI (all else at median)
ax = axes[1, 2]
ndvi_range = np.linspace(df["NDVI"].min(), df["NDVI"].max(), 100)
X_pd = np.tile(np.median(X_tr, axis=0), (100, 1))
ndvi_col = FEATURES.index("NDVI")
ndbi_col = FEATURES.index("NDBI")
X_pd[:, ndvi_col] = (ndvi_range - scaler.mean_[ndvi_col]) / scaler.scale_[ndvi_col]

cnn_pd_ndvi = predict_cnn(best_cnn_model, best_cnn_device, X_pd)

ndbi_range = np.linspace(df["NDBI"].min(), df["NDBI"].max(), 100)
X_pd2 = np.tile(np.median(X_tr, axis=0), (100, 1))
X_pd2[:, ndbi_col] = (ndbi_range - scaler.mean_[ndbi_col]) / scaler.scale_[ndbi_col]
cnn_pd_ndbi = predict_cnn(best_cnn_model, best_cnn_device, X_pd2)

ax.plot(ndvi_range, cnn_pd_ndvi, color=P["green"], lw=2.5, label="NDVI effect")
ax.plot(ndbi_range, cnn_pd_ndbi, color=P["warm"],  lw=2.5, label="NDBI effect")
ax.legend(fontsize=8, facecolor=P["panel"], labelcolor=P["text"])
sax(ax, "CNN Partial Dependence\n(all other features at median)",
    "Index Value", "Predicted LST (°C)")

fig.suptitle("Core Research Question — How NDVI & NDBI Drive LST Variations",
             color=P["text"], fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_rq_analysis.png", dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close()
print("Research question analysis figure saved → fig_rq_analysis.png")


=== NDVI & NDBI ISOLATION ANALYSIS ===
Research question analysis figure saved → fig_rq_analysis.png


In [14]:
# 8. FINAL SUMMARY
print("====== FINAL MODEL RANKING ======")
print(res_df_sorted.drop(columns=["CV_RMSE"]).to_string(index=False))

best_model = res_df_sorted.iloc[0]
print(f"\nBest model : {best_model['Model']}")
print(f"   RMSE       : {best_model['RMSE']:.4f} °C")
print(f"   MAE        : {best_model['MAE']:.4f} °C")
print(f"   R²         : {best_model['R2']:.4f}")
print(f"   MAPE       : {best_model['MAPE']:.3f} %")

print(f"\nNDVI–LST correlation  : r = {r_ndvi:.4f}  (negative → cooling effect)")
print(f"NDBI–LST correlation  : r = {r_ndbi:.4f}  (positive → heating effect)")

print("\nOutput files:")
for f in ["fig_eda.png", "fig_eval.png", "fig_hotspot.png", "fig_rq_analysis.png"]:
    print(f"  • {f}")
print("\nPipeline complete.")

====== FINAL MODEL RANKING ======
            Model     RMSE      MAE       R2     MAPE
Gradient Boosting 1.257618 0.973766 0.841612 3.152390
    Random Forest 1.297283 1.011393 0.831464 3.280449
              MLP 1.458494 1.139987 0.786974 3.709696
 1D-CNN (PyTorch) 1.688865 1.314897 0.714363 4.171184
Linear Regression 1.721434 1.338111 0.703240 4.415952

Best model : Gradient Boosting
   RMSE       : 1.2576 °C
   MAE        : 0.9738 °C
   R²         : 0.8416
   MAPE       : 3.152 %

NDVI–LST correlation  : r = -0.6131  (negative → cooling effect)
NDBI–LST correlation  : r = 0.6877  (positive → heating effect)

Output files:
  • fig_eda.png
  • fig_eval.png
  • fig_hotspot.png
  • fig_rq_analysis.png

Pipeline complete.
